<a href="https://colab.research.google.com/github/Harshithpalan/Python-projects/blob/main/Few-Shot%20Learning%20Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Few-Shot Learning with Siamese Networks

Siamese Networks consist of two identical neural networks that share the same weights. They are used to find the similarity of the inputs by comparing their feature vectors. This is particularly useful for few-shot learning tasks like face recognition or signature verification.

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models, Model
import numpy as np
import matplotlib.pyplot as plt

def create_base_network(input_shape):
    # A simple CNN to extract features
    input = layers.Input(shape=input_shape)
    x = layers.Conv2D(64, (3, 3), activation='relu')(input)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, (3, 3), activation='relu')(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu')(x)
    return Model(input, x)

input_shape = (28, 28, 1)
base_network = create_base_network(input_shape)

# Create the Siamese Network
input_a = layers.Input(shape=input_shape)
input_b = layers.Input(shape=input_shape)

# Both inputs pass through the same network
feat_a = base_network(input_a)
feat_b = base_network(input_b)

# Calculate L1 distance between the two feature vectors
distance = layers.Lambda(lambda tensors: tf.abs(tensors[0] - tensors[1]))([feat_a, feat_b])
outputs = layers.Dense(1, activation='sigmoid')(distance)

siamese_model = Model(inputs=[input_a, input_b], outputs=outputs)
siamese_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

siamese_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 28, 28, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_2       │ (None, 28, 28, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional          │ (None, 128)       │  2,057,088 │ input_layer_1[0]… │
│ (Functional)        │                   │            │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 128)       │          0 │ functional[0][0], │
│                     │                   │            │ functional[1][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1)         │        129 │ lambda[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,057,217 (7.85 MB)

 Trainable params: 2,057,217 (7.85 MB)

 Non-trainable params: 0 (0.00 B)

### Next Steps:
1. **Data Preparation**: You will need to create pairs of images (positive pairs from the same class, negative pairs from different classes).
2. **Training**: Train the model to output 1 for similar pairs and 0 for dissimilar pairs.
3. **Inference**: Use the learned embeddings to classify new, unseen classes with just one or two examples.